# Using the DispatchableComposite Module in baseobjects.composition

## Introduction

`DispatchableComposite` is a powerful base class that combines the functionality of `BaseDispatchingComposite` and `DispatchableClass`. It allows for:
1. **Class Dispatching**: Automatically selecting the correct subclass to instantiate based on input arguments.
2. **Component Dispatching**: Dynamically determining which components to create for the selected instance.

This is particularly useful for implementing factory-like patterns where the "Head Class" acts as a dispatcher that returns specialized composite objects tailored to specific needs.

This tutorial covers:
- Setting up a `DispatchableComposite` hierarchy
- Implementing class dispatching with `get_registered_class`
- Implementing component dispatching with `dispatch_component_types`
- Using the combined dispatching mechanism

**Prerequisites:**
- Familiarity with `BaseDispatchingComposite`
- Understanding of `DispatchableClass` and `NamespaceClassRegistry`
- Installed package: `baseobjects`

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

We need `DispatchableComposite`, `BaseComponent`, and `NamespaceClassRegistry`.


In [ ]:
from typing import Any, ClassVar
from baseobjects.composition import DispatchableComposite, BaseComponent
from baseobjects.classregistration import NamespaceClassRegistry


## Core Functionality

`DispatchableComposite` leverages both class registration and component dispatching to provide a highly dynamic instantiation process.

### Key Concepts

1. **Class Dispatching**: The `@classmethod get_registered_class` determines which subclass is actually instantiated.
2. **Component Dispatching**: The `dispatch_component_types` method determines which components are attached to the instance.

### Defining Components

We'll use some simple components for a data pipeline example.


In [ ]:
class DataSource(BaseComponent):
    def get_data(self): return "base data"

class FileSource(DataSource):
    def __init__(self, *args, path="data.txt", **kwargs):
        super().__init__(*args, **kwargs)
        self.path = path
    def get_data(self): return f"data from {self.path}"

class ApiSource(DataSource):
    def __init__(self, *args, url="http://api.com", **kwargs):
        super().__init__(*args, **kwargs)
        self.url = url
    def get_data(self): return f"data from {self.url}"


### Defining the Dispatchable Hierarchy

We'll create a `DataPipeline` that can dispatch to different specialized pipeline types.


In [ ]:
class DataPipeline(DispatchableComposite):
    """Base pipeline that dispatches to subclasses."""
    class_registration = True
    class_registry = NamespaceClassRegistry()

    @classmethod
    def get_registered_class(cls, pipeline_type: str = "base", **kwargs) -> type["DataPipeline"] | None:
        """Determines which subclass to use."""
        return cls.class_registry.get_class("pipeline", pipeline_type)

    def dispatch_component_types(self, source_type: str = "file", **kwargs) -> dict[str, tuple[type, dict[str, Any]]]:
        """Determines which components to create."""
        if source_type == "file":
            return {"source": (FileSource, {"path": kwargs.get("path", "default.txt")})}
        elif source_type == "api":
            return {"source": (ApiSource, {"url": kwargs.get("url", "http://default.com")})}
        return {"source": (DataSource, {})}

    def run(self):
        data = self.components["source"].get_data()
        print(f"[{type(self).__name__}] Running with: {data}")

class ETLPipeline(DataPipeline):
    """A specialized pipeline for ETL."""
    pass

class AnalyticsPipeline(DataPipeline):
    """A specialized pipeline for Analytics."""
    pass

# Register the subclasses
DataPipeline.class_registry.register_class(ETLPipeline, namespace="pipeline", name="etl")
DataPipeline.class_registry.register_class(AnalyticsPipeline, namespace="pipeline", name="analytics")
DataPipeline.class_registry.register_class(DataPipeline, namespace="pipeline", name="base")


### Using Combined Dispatching

When you instantiate `DataPipeline`, it first selects the class, then builds the components.


In [ ]:
# 1. Create an ETL pipeline with a File source
etl_p = DataPipeline(pipeline_type="etl", source_type="file", path="raw_data.csv")
etl_p.run()
print(f"Instance type: {type(etl_p)}")

# 2. Create an Analytics pipeline with an API source
ana_p = DataPipeline(pipeline_type="analytics", source_type="api", url="http://weather.api")
ana_p.run()
print(f"Instance type: {type(ana_p)}")

# 3. Create the base pipeline
base_p = DataPipeline(pipeline_type="base", source_type="none")
base_p.run()


## Module Interaction

`DispatchableComposite` coordinates between the class registration system (for deciding the instance type) and the composition system (for building the internal structure). It ensures that the arguments passed during instantiation are correctly routed to both the class selection logic and the component creation logic.


## Advanced Features

### Dynamic Re-configuration

Since `DispatchableComposite` inherits from `BaseDispatchingComposite`, you can still use `construct` to change or add components after the initial dispatch.


In [ ]:
# Re-configure the base pipeline to use an API source
base_p.construct(source_type="api", url="http://new-source.api")
base_p.run()


## Examples

Implementing a multi-format report generator where the report type (PDF, HTML) determines the class, and the data source determines the components.


In [ ]:
class ReportGenerator(DispatchableComposite):
    class_registration = True
    class_registry = NamespaceClassRegistry()
    @classmethod
    def get_registered_class(cls, fmt="text", **kwargs):
        return cls.class_registry.get_class("report", fmt)
    def dispatch_component_types(self, **kwargs):
        return {"data": (DataSource, {})}
    def generate(self): pass

class PDFReport(ReportGenerator):
    def generate(self): print("Generating PDF...")

ReportGenerator.class_registry.register_class(PDFReport, "report", "pdf")

rep = ReportGenerator(fmt="pdf")
rep.generate()


## API Highlights

- **`DispatchableComposite`**: The ultimate dynamic composite class.
  - `get_registered_class(*args, **kwargs)`: Class method for type dispatching.
  - `dispatch_component_types(*args, **kwargs)`: Instance method for component dispatching.
  - `class_registry`: Registry used for class selection.

## Troubleshooting / FAQs

- **Problem**: `TypeError: 'NoneType' object is not callable` during instantiation.
  - **Solution**: `get_registered_class` might be returning `None`. Ensure that the arguments passed match a registered class in the `class_registry`.

- **Problem**: Arguments are not reaching the component's `__init__`.
  - **Solution**: Ensure that `dispatch_component_types` correctly extracts them from `kwargs` and places them in the component's options dictionary.

## Conclusion and Next Steps

`DispatchableComposite` is the most advanced composite base in `baseobjects`. It provides a complete framework for building dynamic, self-configuring object hierarchies that can adapt to a wide variety of runtime requirements.

- **Next**: Check out `CompositeFactoryClass` for a preset-based approach to similar problems.
- **Reference**: See `src/baseobjects/composition/dispatchablecomposite.py` for implementation details.

`DispatchableComposite` is the most advanced composite base in `baseobjects`. It provides a complete framework for building dynamic, self-configuring object hierarchies that can adapt to a wide variety of runtime requirements.
